In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import gridspec
from matplotlib.patches import Rectangle, Patch
import plotly.graph_objects as go
from PIL import Image
from tqdm.auto import tqdm
import os
import warnings
import webbrowser
import json

# Import custom modules
from macrohet import visualise, colours

# Constants
MICROMETERS_PER_PIXEL = 0.14949402  # Converting 1.4949402023919043E-07 meters to micrometers

# Set plotting defaults
plt.rcParams.update({
    'font.family': 'Nimbus Sans',
    'font.size': 14
})

# Apply custom visualization defaults
visualise.set_plotting_defaults()

# Set color palette
expanded_piyg = colours.expanded_piyg

# Optional: Set common Plotly template for consistency
plotly_template = {
    'layout': {
        'font': {'family': 'Nimbus Sans', 'size': 14},
        'plot_bgcolor': 'white',
        'paper_bgcolor': 'white',
        'margin': {'t': 40, 'b': 40, 'l': 40, 'r': 40}
    }
}

Helvetica not found. Using Nimbus Sans font.
Default plotting aesthetic loaded.


In [2]:
# df = pd.read_pickle('/Volumes/OPERA2/Nathan/macrohet_syno/results/dfs/sc_df.pkl')
df = pd.read_pickle('/mnt/SYNO/macrohet_syno/results/dfs/sc_df.pkl')


In [3]:
df

,Time (hours),Mtb Area (µm),dMtb Area (µm),Mphi Area (µm),dMphi Area (µm),Infection Status,Initial Infection Status,Final Infection Status,x,y,...,dMtb Area between frames (µm),Mtb Area Processed (µm),Time Model (hours),Mtb Area Model (µm),mtb_origin,Doubling Amounts,Doubling Times,r2,Frame,category_rank
0,0.0,0.000000,0.0,4649.351562,2080.574775,NaN,0.0,0.0,107.845200,755.241882,...,NaN,NaN,NaN,NaN,None,None,None,1.0,0,NaN
1,0.5,0.000000,0.0,4973.001953,2080.574775,NaN,0.0,0.0,113.264671,764.392212,...,0.0,NaN,NaN,NaN,None,None,None,1.0,1,NaN
2,1.0,0.000000,0.0,4687.857910,2080.574775,NaN,0.0,0.0,116.562256,767.332825,...,0.0,NaN,NaN,NaN,None,None,None,1.0,2,NaN
3,1.5,0.000000,0.0,4209.064453,2080.574775,NaN,0.0,0.0,111.890106,758.421265,...,0.0,NaN,NaN,NaN,None,None,None,1.0,3,NaN
4,2.0,0.000000,0.0,4061.855225,2080.574775,False,0.0,0.0,114.113556,759.828735,...,0.0,NaN,0.0,0.000000,None,None,None,1.0,4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
604938,74.5,1.318559,0.0,465.719601,-259.532690,1.0,0.0,0.0,199.740707,530.029724,...,NaN,3.754542,70.0,3.831754,NaN,None,None,0.92,149,8.0
604939,75.0,0.000000,0.0,392.506040,-259.532690,0.0,0.0,0.0,200.346191,529.733398,...,NaN,2.771209,70.5,3.993013,NaN,None,None,0.92,150,8.0
604940,75.5,0.000000,0.0,410.295415,-259.532690,0.0,0.0,0.0,200.246902,528.299072,...,NaN,3.285224,71.0,4.152034,NaN,None,None,0.92,151,8.0
604941,76.0,0.000000,0.0,463.708240,-259.532690,0.0,0.0,0.0,200.559875,529.950134,...,NaN,5.251889,71.5,4.311997,NaN,None,None,0.92,152,8.0


### Params

In [4]:
### set plotting parameters
# Input figure size in millimeters
width_mm = 280  # for example, 180 mm
height_mm = 100  # for example, 80 mm
# Convert millimeters to inches (1 inch = 25.4 mm)
width_in = width_mm / 25.4
height_in = height_mm / 25.4

### set biological parameters
# Step 1: Bin the Doubling Times into categories
bins = [0, 16, 24, float('inf')]  # Define the bins: <20, 20-24, >24
labels = ['Fast', 'Normal', 'Slow']  # Labels for each bin
# Define the desired order with CTRL at the top
strain_order = ['CTRL', 'PZA', 'INH', 'RIF']

# F1 interactive

In [5]:
# Filter the subset DataFrame based on conditions
IDs = df[
    (df['Compound'] == 'CTRL') &
    (df['Strain'] == 'WT') &
    (df['mtb_origin'] != 'Junk') &
    (df['Infection Status'] == True)
].ID.unique()

# Ensure that df contains only rows with IDs present in the subset_df
subset_df = df[df['ID'].isin(IDs)]

In [6]:
# add image links
subset_df['Image Link'] = subset_df['Image Link'] = subset_df.apply(
    # lambda row: f"https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/{row['ID']}_t{row['Time (hours)']}.png",
    # lambda row: f"https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/glimpse_frames/4108.4.3.ND0003/frame_067.png",
    lambda row: f"https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/glimpse_frames/{row['ID']}/frame_{int(row['Time (hours)']):03d}.png",
    axis=1
)

In [7]:
subset_df['Image Link'].iloc[0]

'https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/glimpse_frames/1.3.5.PS0000/frame_000.png'

### Prepare data for Plotly

In [8]:


traces = []
cmap = plt.get_cmap('PiYG_r')
norm = mpl.colors.Normalize(vmin=0, vmax=300)

for ID in tqdm(subset_df['ID'].unique()):
    sc_df = subset_df[subset_df['ID'] == ID]
    time_model = sc_df['Time Model (hours)'].dropna().values - min(sc_df['Time Model (hours)'].dropna().values)
    population_model = sc_df['Mtb Area Model (µm)'].dropna().values
    compound = sc_df['Compound'].iloc[0]
    concentration = sc_df['Concentration'].iloc[0] if not 'EC0' else 'N/A'
    # Filter based on max population value
    max_value = np.nanmax(population_model)
    if max_value < 4 or max_value > 300:
        continue

    # Calculate the color value based on the final value of the population model
    colour_value = population_model[-1]
    color = f"rgba{tuple((np.array(cmap(norm(colour_value))) * 255).astype(int))}"

    # Add a trace for each line
    traces.append(go.Scatter(
        x=time_model,
        y=population_model,
        mode='lines',
        line=dict(color=color, width=4),
        name=f"ID: {ID}",  # Line name for identification (optional)
        hoverinfo="text",
        customdata=sc_df['Image Link'],  # Pass image links via custom data
        text=[f"<b>ID:</b> {ID}<br><b>Time:</b> {t:.2f} hours<br><b>Mtb Area:</b> {p:.2f} µm²"
              for t, p in zip(time_model, population_model)],
        opacity=0.6,
        showlegend=False
    ))


  0%|          | 0/2436 [00:00<?, ?it/s]

In [ ]:
# Create a special trace just for the colorbar
# This trace has no visible points but controls the colorbar appearance
colorbar_trace = go.Scatter(
    x=[None],  # Empty data since we only want the colorbar
    y=[None],
    mode='markers',
    marker=dict(
        size=0,  # Make markers invisible
        cmax=300,  # Maximum value for color scale
        cmin=0,   # Minimum value for color scale
        colorbar=dict(
            thickness=15,    # Width of the colorbar
            len=0.9,        # Length of colorbar as fraction of plot height
            outlinewidth=0,  # Remove border around colorbar
            tickfont=dict(family="Helvetica", size=16),  # Style the tick labels
            title=dict(
                text="\nFinal Mtb Load (µm²)",  # Colorbar title with units
                side='right',                   # Position title on right side
                font=dict(family="Helvetica", size=16),
            ),
            xpad=10,  # Add padding between plot and colorbar
        ),
        colorscale="PiYG_r"  # Use PiYG (Pink to Green) color palette
    ),
    hoverinfo='none',    # Disable hover effects
    showlegend=False     # Hide from legend
)

# Combine the colorbar trace with our existing data traces
fig = go.Figure(data=traces + [colorbar_trace])
fig.update_traces(visible=True)  # Ensure all traces are visible

# Configure the overall plot layout
fig.update_layout(
    # X-axis settings
    xaxis=dict( 
        title=dict(text="Time (hours)"),
        gridcolor='rgba(200,200,200,0.3)',  # Light grey grid
        showline=False,   # Hide axis line
        zeroline=False,   # Hide zero line
        showgrid=False,   # Hide grid
        range=[-2, 70],   # Set axis range with padding
    ),
    
    # Y-axis settings
    yaxis=dict(
        title=dict(text="Mtb Area (µm²)"),
        gridcolor='rgba(200,200,200,0.3)',
        showline=False,
        showgrid=False,
        zeroline=False,
        range=[-30, 300],
    ),
    
    # Add custom axis lines offset from the plot
    shapes=[
        # Y-axis line
        dict(
            type="line",
            xref="x", yref="y",
            x0=-1, x1=-1,     # Position slightly left of data
            y0=0, y1=600,     # Extend beyond data range
            line=dict(color="black", width=2)
        ),
        # X-axis line
        dict(
            type="line",
            xref="x", yref="y",
            x0=0, x1=70,      # Span full width
            y0=-20, y1=-20,   # Position slightly below data
            line=dict(color="black", width=2)
        )
    ],
    
    # General plot settings
    hovermode="closest",         # Show hover info for nearest point
    template="plotly_white",     # Use white background template
    autosize=True,              # Enable responsive sizing
    height=None,                # Allow height to adjust automatically
    width=None,                 # Allow width to adjust automatically
    
    # Font settings
    font=dict(
        family="Helvetica",
        size=16
    ),
    
    # Margin settings
    margin=dict(
        l=36,    # Left margin
        r=18,    # Right margin
        t=36,    # Top margin
        b=36,    # Bottom margin
        pad=6    # Internal padding
    )
)

In [19]:
# Save with a simpler HTML template that allows popups to escape
html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        .container {{
            width: 100%;
            position: relative;
        }}
        #plot {{
            width: 100%;
            height: 600px;
        }}
        .hover-popup {{
            position: fixed !important;
            z-index: 1000;
            width: 300px;  /* Fixed width */
            background-color: white;
            border: 2px solid white;
            padding: 5px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            display: flex;
            align-items: center;
        }}
        .hover-popup img {{
            width: 300px;  /* Force image to match container width */
            height: auto;
        }}
    </style>
</head>
<body>
    <div class="container">
        <div id="plot"></div>
    </div>
    <script>
        const figure = {fig.to_json()};
        Plotly.newPlot('plot', figure.data, figure.layout);
        
        document.getElementById('plot').on('plotly_hover', function(eventData) {{
            const point = eventData.points[0];
            if (point.customdata) {{
                const hoverContent = document.createElement('div');
                hoverContent.className = 'hover-popup';
                hoverContent.innerHTML = `<img src="${{point.customdata}}">`;
                document.body.appendChild(hoverContent);
                
                // Simple offset from cursor, no boundary checking
                hoverContent.style.left = `${{eventData.event.clientX - 310}}px`;
                hoverContent.style.top = `${{eventData.event.clientY + 10}}px`;
                
                document.getElementById('plot').on('plotly_unhover', function() {{
                    hoverContent.remove();
                }});
            }}
        }});
    </script>
</body>
</html>
"""

# Write the file
output_file = "interactive_plots/F1H.html"
with open(output_file, "w") as f:
    f.write(html_template)

# After your existing code that saves the file:
webbrowser.open('file://' + os.path.realpath(output_file))

True

# The rest of F1

In [14]:
styles = """
    .container {
        width: 100%;
        margin: 0 auto;
    }
    
    /* Header and main image layout */
    .header-image-container {
        position: relative;
        width: 100%;
        margin-bottom: 20px;
    }
    .header-image {
        width: 100%;
        height: auto;
    }
    .methods-link {
            position: absolute;
            top: 10%; /* Position in the top 10% of the image */
            left: 50%; /* Center horizontally */
            transform: translateX(-50%);
            background-color: rgba(255, 255, 255, 0.8); /* Semi-transparent background */
            padding: 5px 10px;
            border-radius: 5px;
            text-decoration: none;
            color: #000;
            font-weight: bold;
            font-size: 14px;
            z-index: 10; /* Ensure it's above the image */
        }
    .methods-link:hover {
        background-color: rgba(255, 255, 255, 1); /* Solid background on hover */
    }
    /* Hover areas and popups */
    .hover-area {
        position: absolute;
        cursor: pointer;
        background: rgba(255, 255, 255, 0.0);
        transition: background 0.3s;
    }
    .hover-area:hover {
        background: rgba(255, 255, 255, 0.1);
    }
    .hover-popup {
        position: fixed !important;
        z-index: 1000;
        background-color: white;
        border: 2px solid white;
        padding: 5px;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        display: flex;
        align-items: center;
    }
    
    /* Plot styling */
    #plot {
        width: 100%;
        aspect-ratio: 22/9;
    }
    
    /* Modal styling */
    .modal {
        display: none;
        position: fixed;
        top: 0;
        left: 0;
        width: 100%;
        height: 100%;
        background-color: rgba(0,0,0,0.9);
        z-index: 2000;
        overflow: hidden;
    }
    .modal-content {
        position: relative;
        width: 90%;
        height: 90%;
        margin: 50px auto;
        display: flex;
        justify-content: center;
        align-items: center;
    }
    .modal-image {
        max-width: 100%;
        max-height: 100%;
        object-fit: contain;
        transform-origin: center;
        cursor: move;
    }
    .close-button {
        position: absolute;
        top: 15px;
        right: 35px;
        color: #f1f1f1;
        font-size: 40px;
        font-weight: bold;
        cursor: pointer;
    }
"""


# Define hover areas HTML
hover_areas = """
    <div class="hover-area" style="left: 2.56%; top: 32.26%; width: 31%; height: 31%;" 
         data-image="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/PS0000.3.5.t0.single_tile.rgba.png"></div>
    <div class="hover-area" style="left: 35.04%; top: 32.26%; width: 31%; height: 31%;" 
         data-image="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/PS0000.3.5.t0.rgba.png"></div>
    <div class="hover-area" style="left: 35.04%; top: 66.89%; width: 31%; height: 31%;" 
         data-image="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/PS0000.3.5.t0.segmented_opaque.rgba.png"></div>
    <div class="hover-area" style="left: 2.56%; top: 66.89%; width: 31%; height: 31%;" 
         data-type="video"
         data-media="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/PS0000_(3%2C+5)_WT_CTRL_0.mp4"></div>
    <div class="hover-area" style="left: 67.52%; top: 66.89%; width: 31%; height: 31%;" 
         data-type="video"
         data-media="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/s1_annotated.mp4"></div>
"""

In [12]:
import json
import numpy as np

def convert_to_serializable(obj):
    """
    Recursively convert non-serializable objects to serializable types
    """
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(v) for v in obj]
    elif hasattr(obj, 'to_plotly_json'):
        return convert_to_serializable(obj.to_plotly_json())
    return obj

# Prep the figure data for serialization
figure_data = {
    'data': [],
    'layout': convert_to_serializable(fig.layout)
}

# Convert each trace
for trace in fig.data:
    converted_trace = convert_to_serializable(trace)
    figure_data['data'].append(converted_trace)

# Write to file
with open('../docs/figures/data/F1H_plot_data_.json', 'w') as f:
    json.dump(figure_data, f, indent=2)

In [ ]:
'../'

In [11]:
print('fufucken done')

fufucken done


In [81]:
# Create the HTML template with Plotly figure and hover functionality
# Update the template with hover functionality
html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <link rel="stylesheet" href="css/styles.css">
</head>
<body>
    <div class="container">
        <div class="header-image-container">
            <img src="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/Figure_1_crop.png" 
                 alt="Header Image" 
                 class="header-image">
                 <a href="../index.html#materials-and-methods" class="methods-link">Go to Methods</a>
                   <div class="hover-area" style="left: 2.56%; top: 32.26%; width: 31%; height: 31%;" 
                         data-image="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/PS0000.3.5.t0.single_tile.rgba.png"></div>
                    <div class="hover-area" style="left: 35.04%; top: 32.26%; width: 31%; height: 31%;" 
                         data-image="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/PS0000.3.5.t0.rgba.png"></div>
                    <div class="hover-area" style="left: 35.04%; top: 66.89%; width: 31%; height: 31%;" 
                         data-image="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/PS0000.3.5.t0.segmented_opaque.rgba.png"></div>
                    <div class="hover-area" style="left: 2.56%; top: 66.89%; width: 31%; height: 31%;" 
                         data-type="video"
                         data-media="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/PS0000_(3%2C+5)_WT_CTRL_0.mp4"></div>
                    <div class="hover-area" style="left: 67.52%; top: 66.89%; width: 31%; height: 31%;" 
                         data-type="video"
                         data-media="https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/figs/misc/s1_annotated.mp4"></div>
            <div id="hover-popup" class="hover-popup">
                <img id="popup-image" style="width: 350px; height: auto;">  <!-- Set initial width -->
            </div>
        </div>
        <div id="plot"></div>
    </div>

    <div id="imageModal" class="modal">
        <span class="close-button">&times;</span>
        <div class="modal-content">
            <img id="modalImage" class="modal-image">
            <video id="modalVideo" class="modal-image" controls>
                Your browser does not support the video tag.
            </video>
        </div>
    </div>

    <script>
        // Configuration variables
        const PLOT_POPUP_WIDTH = 350; 
        
        // Initialize the Plotly plot
        const figure = {fig.to_json()};
        Plotly.newPlot('plot', figure.data, figure.layout);
        
        // Add hover functionality for plot
        document.getElementById('plot').on('plotly_hover', function(eventData) {{
            const point = eventData.points[0];
            if (point.customdata) {{
                const hoverContent = document.createElement('div');
                hoverContent.className = 'hover-popup';
                hoverContent.innerHTML = `<img src="${{point.customdata}}" style="width: ${{PLOT_POPUP_WIDTH}}px; height: auto;">`;
                document.body.appendChild(hoverContent);
                
                hoverContent.style.left = `${{eventData.event.clientX - PLOT_POPUP_WIDTH - 10}}px`;
                hoverContent.style.top = `${{eventData.event.clientY + 10}}px`;
                
                document.getElementById('plot').on('plotly_unhover', function() {{
                    hoverContent.remove();
                }});
            }}
        }});

        // Set up hover areas functionality
        const hoverAreas = document.querySelectorAll('.hover-area');
        const hoverPopup = document.getElementById('hover-popup');
        const popupImage = document.getElementById('popup-image');
        const modal = document.getElementById('imageModal');
        const modalImg = document.getElementById('modalImage');
        const closeButton = document.querySelector('.close-button');
        const modalVideo = document.getElementById('modalVideo');

        let scale = 1;
        let panning = false;
        let pointX = 0;
        let pointY = 0;
        let start = {{ x: 0, y: 0 }};

        hoverAreas.forEach(area => {{
            let isHovering = false;
        
            area.addEventListener('mousemove', (e) => {{
                isHovering = true;
                const mediaType = area.getAttribute('data-type') || 'image';
                const mediaUrl = mediaType === 'video' ? 
                    area.getAttribute('data-media') : 
                    area.getAttribute('data-image');
                
                if (mediaType === 'video') {{
                    hoverPopup.style.display = 'block';
                    if (!popupImage.parentElement.querySelector('video')) {{
                        const video = document.createElement('video');
                        video.src = mediaUrl;
                        video.style.width = `${{PLOT_POPUP_WIDTH}}px`;
                        video.style.height = 'auto';
                        video.autoplay = true;
                        video.loop = true;
                        video.muted = true;
                        popupImage.parentElement.appendChild(video);
                        popupImage.style.display = 'none';
                    }}
                }} else {{
                    popupImage.src = mediaUrl;
                    popupImage.style.display = 'block';
                    const video = popupImage.parentElement.querySelector('video');
                    if (video) video.remove();
                }}
                
                hoverPopup.style.display = 'block';
                hoverPopup.style.left = `${{e.clientX - PLOT_POPUP_WIDTH - 10}}px`;
                hoverPopup.style.top = `${{e.clientY + 10}}px`;
            }});
        
            area.addEventListener('mouseleave', () => {{
                isHovering = false;
                hoverPopup.style.display = 'none';
                const video = popupImage.parentElement.querySelector('video');
                if (video) video.remove();
                popupImage.style.display = 'block';
            }});
        
            // Replace the old click handler with this new one
            if (!area.hasAttribute('data-type')) {{
                // Image handling
                area.addEventListener('click', (e) => {{
                    if (isHovering) {{
                        modal.style.display = 'block';
                        modalImg.style.display = 'block';
                        modalVideo.style.display = 'none';
                        modalImg.src = popupImage.src;
                        scale = 1;
                        pointX = 0;
                        pointY = 0;
                        modalImg.style.transform = `scale(${{scale}}) translate(${{pointX}}px, ${{pointY}}px)`;
                    }}
                }});
            }} else if (area.getAttribute('data-type') === 'video') {{
                // Video handling
                area.addEventListener('click', (e) => {{
                    if (isHovering) {{
                        modal.style.display = 'block';
                        modalImg.style.display = 'none';
                        modalVideo.style.display = 'block';
                        modalVideo.src = area.getAttribute('data-media');
                        modalVideo.play();
                    }}
                }});
            }}
        }});

        // Modal functionality
        closeButton.addEventListener('click', function() {{
            modal.style.display = 'none';
        }});

        // Zoom with mouse wheel
        modalImg.addEventListener('wheel', function(e) {{
            e.preventDefault();
            
            // Get dimensions of image and container
            const rect = modalImg.getBoundingClientRect();
            const container = modal.getBoundingClientRect();
            
            // Get mouse position relative to container center
            const mouseX = e.clientX - (container.left + container.width / 2);
            const mouseY = e.clientY - (container.top + container.height / 2);
            
            // Calculate new scale
            const oldScale = scale;
            if (e.deltaY < 0) {{
                scale = Math.min(scale * 1.1, 4);
            }} else {{
                scale = Math.max(scale / 1.1, 0.125);
            }}
            
            // Update position to maintain mouse point position
            pointX = mouseX * (1 - scale);
            pointY = mouseY * (1 - scale);
            
            modalImg.style.transformOrigin = 'center';
            modalImg.style.transform = `scale(${{scale}}) translate(${{pointX / scale}}px, ${{pointY / scale}}px)`;
        }});
        
        // Pan functionality
        modalImg.addEventListener('mousedown', function(e) {{
            e.preventDefault();
            start = {{ x: e.clientX - pointX, y: e.clientY - pointY }};
            panning = true;
        }});
        
        document.addEventListener('mousemove', function(e) {{
            if (!panning) return;
            pointX = (e.clientX - start.x);
            pointY = (e.clientY - start.y);
            modalImg.style.transform = `scale(${{scale}}) translate(${{pointX / scale}}px, ${{pointY / scale}}px)`;
        }});

        document.addEventListener('mouseup', function() {{
            panning = false;
        }});

        modal.addEventListener('mousedown', function(e) {{
            if (e.target === modal) {{
                modal.style.display = 'none';
                scale = 1;
                pointX = 0;
                pointY = 0;
                modalImg.style.transform = `scale(1) translate(0px, 0px)`;
            }}
        }});
    </script>
</body>
</html>
"""
# Save the plot to an HTML file with added video functionality
output_file = "interactive_plots/F1.html"
# Write the HTML file
with open(output_file, "w") as f:
    f.write(html_template)
print(f"Interactive plot with video functionality saved to {output_file}")

# After your existing code that saves the file:
webbrowser.open('file://' + os.path.realpath(output_file))

Interactive plot with video functionality saved to interactive_plots/F1.html


True

# Save out into true folder now

In [82]:
# Save the plot to an HTML file with added video functionality
output_file = "../docs/figures/F1.html"
# Write the HTML file
with open(output_file, "w") as f:
    f.write(html_template)
print(f"Interactive plot with video functionality saved to {output_file}")
# After your existing code that saves the file:
webbrowser.open('file://' + os.path.realpath('../docs/index.html'), new=2)

Interactive plot with video functionality saved to ../docs/figures/F1.html


True

# FIG 1H for antibiotics

In [221]:
import random

In [224]:
# Filter the subset DataFrame based on conditions
IDs = df[
    # (df['Compound'] == 'CTRL') &
    # (df['Strain'] == 'WT') &
    (df['mtb_origin'] != 'Junk') 
    # (df['Infection Status'] == True)
].ID.unique()
random.shuffle(IDs)


In [227]:
# add image links
subset_df['Image Link'] = subset_df['Image Link'] = subset_df.apply(
    # lambda row: f"https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/{row['ID']}_t{row['Time (hours)']}.png",
    lambda row: f"https://s3.eu-west-2.amazonaws.com/macrohet.glimpses/glimpse_frames/4108.4.3.ND0003/frame_067.png",
    # lambda row: f"https://macrohet.s3.eu-west-2.amazonaws.com/macrohet.glimpses/glimpse_frames/{ID}/frame_{int(row['Time (hours)']):03d}.png",
    axis=1
)

In [228]:
len(IDs)

12563

In [262]:
# Ensure that df contains only rows with IDs present in the subset_df
subset_df = df[df['ID'].isin(IDs)]

In [ ]:

# Create figure with vertical subplots
fig = make_subplots(
    rows=2, 
    cols=1,
    subplot_titles=["EC50", "EC99"],
    shared_xaxes=True,  # Share x axes for alignment
    vertical_spacing=0.1  # Add some space between plots
)

def optimize_timeseries(time_values, population_values, target_points=35):
    """Reduce number of points in timeseries while preserving shape"""
    if len(time_values) <= target_points:
        return time_values, population_values
    
    step = max(1, len(time_values) // target_points)
    return time_values[::step], population_values[::step]

# Pre-process control data
ctrl_data = subset_df[(subset_df['Compound'] == 'CTRL') & (subset_df['Concentration'] == 'EC0')]
max_value = 0

# Plot data for each concentration subplot
for row, concentration in enumerate(treatment_concentrations, 1):
    # First plot control data in this subplot
    for i, ID in enumerate(ctrl_data['ID'].unique()):
        sc_df = ctrl_data[ctrl_data['ID'] == ID]
        
        time_model = sc_df['Time Model (hours)'].dropna().values
        if len(time_model) > 0:
            time_model = time_model - min(time_model)
            population_model = sc_df['Mtb Area Model (µm)'].dropna().values
            
            # Optimize the number of points
            time_model, population_model = optimize_timeseries(time_model, population_model)
            
            if np.nanmax(population_model) > max_value:
                max_value = np.nanmax(population_model)
            
            # Only show legend for the very first control trace
            show_legend = (i == 0 and row == 1)
            
            fig.add_trace(
                go.Scatter(
                    x=time_model,
                    y=population_model,
                    mode='lines',
                    line=dict(
                        color=drug_colors['CTRL'],
                        width=2,
                        dash='dot'
                    ),
                    name='CTRL',
                    legendgroup='CTRL',
                    showlegend=show_legend,
                    hovertemplate=(
                        "<b>Drug:</b> CTRL<br>" +
                        "<b>Time:</b> %{x:.1f} hours<br>" +
                        "<b>Mtb Area:</b> %{y:.1f} µm²<br>" +
                        "<b>Concentration:</b> EC0<br>" +
                        f"<b>ID:</b> {ID}<extra></extra>"
                    ),
                    opacity=0.4
                ),
                row=row,
                col=1
            )
    
    # Then plot the treatment data for this concentration
    conc_data = subset_df[subset_df['Concentration'] == concentration]
    
    for drug in treatment_drugs:
        drug_data = conc_data[conc_data['Compound'] == drug]
        first_drug_trace = True
        
        for ID in drug_data['ID'].unique():
            sc_df = drug_data[drug_data['ID'] == ID]
            
            time_model = sc_df['Time Model (hours)'].dropna().values
            if len(time_model) > 0:
                time_model = time_model - min(time_model)
                population_model = sc_df['Mtb Area Model (µm)'].dropna().values
                
                # Optimize the number of points
                time_model, population_model = optimize_timeseries(time_model, population_model)
                
                if np.nanmax(population_model) > max_value:
                    max_value = np.nanmax(population_model)
                
                fig.add_trace(
                    go.Scatter(
                        x=time_model,
                        y=population_model,
                        mode='lines',
                        line=dict(
                            color=drug_colors[drug],
                            width=2
                        ),
                        name=drug,
                        legendgroup=drug,
                        showlegend=(first_drug_trace and row == 1),
                        hovertemplate=(
                            f"<b>Drug:</b> {drug}<br>" +
                            "<b>Time:</b> %{x:.1f} hours<br>" +
                            "<b>Mtb Area:</b> %{y:.1f} µm²<br>" +
                            f"<b>Concentration:</b> {concentration}<br>" +
                            f"<b>ID:</b> {ID}<extra></extra>"
                        ),
                        opacity=0.7
                    ),
                    row=row,
                    col=1
                )
                first_drug_trace = False

# Calculate dimensions for 16:9 aspect ratio
plot_width = 1000  # Base width
plot_height = int((plot_width / 16) * 9)  # Height for one subplot
total_height = int(plot_height * 2.2)  # Total height for both subplots plus spacing

# Update layout with 16:9 aspect ratio and legend on top
fig.update_layout(
    height=total_height,
    width=plot_width,
    template="plotly_white",
    showlegend=True,
    legend=dict(
        orientation="h",  # Horizontal legend
        yanchor="bottom",
        y=1.12,  # Position above plots
        xanchor="center",
        x=0.5,
        title=dict(text="Drug", font=dict(family="Helvetica", size=14)),
        font=dict(family="Helvetica", size=12)
    ),
    margin=dict(l=60, r=20, t=100, b=60),  # Increased top margin for legend
    uirevision=True,
    hovermode='closest'
)

# Update y-axis titles
fig.update_yaxes(title_text="Mtb Area (µm²)", row=1, col=1)
fig.update_yaxes(title_text="Mtb Area (µm²)", row=2, col=1)

# Update x-axis title (only show on bottom plot)
fig.update_xaxes(title_text="Time (hours)", row=2, col=1)

In [266]:
# Save as standalone HTML file
fig.write_html(
    "../interactive_plots/mtb_growth_curves.html",
    include_plotlyjs=True,  # Include Plotly.js in the file
    full_html=True,  # Create a standalone HTML file
    include_mathjax=False,  # Don't include MathJax unless you need equations
    config={
        'responsive': True,  # Make the plot responsive
        'displayModeBar': True,  # Show the modebar with download options
        'toImageButtonOptions': {
            'format': 'png',  # Default image download format
            'filename': 'mtb_growth_curves',
            'height': None,
            'width': None,
            'scale': 3  # Higher resolution for downloads
        }
    }
)
